<a href="https://colab.research.google.com/github/aviral911/DataScienceAndVisualization/blob/main/Time_Series_Tools_and_Basics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Time Series Data Analysis — Tools and Basics
**Python | Pandas | Beginner practical lecture**

Based on the topic sequence in Wes McKinney, *Python for Data Analysis*, 3rd edition, [Chapter 11, sections 11.1 and 11.2](https://wesmckinney.com/book/time-series). Explanations and retail examples are original. Run cells from top to bottom in Jupyter or Google Colab; no external dataset download is needed.

### Learning outcomes
- Understand date, time, datetime and duration objects.
- Convert dates between strings and datetime types; identify missing dates.
- Create a time-indexed Series and DataFrame.
- Select observations by date, month, year and date interval.
- Align time series and aggregate repeated timestamps.

**Scope:** Date ranges, frequencies, shifting, time zones, period arithmetic, resampling and moving windows are reserved for later lessons.

## 1. What is a time series?
A time series is a collection of measurements associated with points in time. For example, a shop records units sold every day. Time is not simply another value: it tells us when an observation occurred and allows chronological comparisons.

| Form | Retail example |
|---|---|
| Timestamp | A sale at 10:30 AM on 1 February 2026 |
| Period | Total sales for February 2026 |
| Interval | A delivery between 10 AM and 11 AM |
| Elapsed time | Minutes since an order was placed |

Observations can occur at regular intervals (daily sales) or irregular intervals (individual customer orders).

In [1]:
import pandas as pd
import numpy as np
from datetime import date, time, datetime, timedelta
from IPython.display import display

## 2. Python date and time tools
Python’s standard library provides `datetime` for dates and times, `time` for clock-related utilities, and `calendar` for calendar operations. We mainly use the classes from `datetime` here.

| Class | Stores |
|---|---|
| `date` | Year, month, day |
| `time` | Hour, minute, second, microsecond |
| `datetime` | Date and time together |
| `timedelta` | Duration between two times |
| `tzinfo` | Interface for time-zone information (covered later) |

In [2]:
shop_date = date(2026, 2, 1)
opening_time = time(10, 30)
order_time = datetime(2026, 2, 1, 10, 30)
print(shop_date)
print(opening_time)
print(order_time)
print(type(order_time))

2026-02-01
10:30:00
2026-02-01 10:30:00
<class 'datetime.datetime'>


### 2.1 Current date and time and individual components
`datetime.now()` reads the runtime clock (Colab may use a different time zone from your location). Attributes such as `.year` and `.hour` extract individual components.

In [3]:
now = datetime.now()
print(now)
print("Year, month, day:", now.year, now.month, now.day)
print("Order hour and minute:", order_time.hour, order_time.minute)

2026-11-25 05:40:46.088050
Year, month, day: 2026 11 25
Order hour and minute: 10 30


### 2.2 Date arithmetic with `timedelta`
Subtract two datetime objects to get a duration. Add a duration to a datetime to estimate a future date. `total_seconds()` includes whole days; `.seconds` only gives the seconds remaining after whole days.

In [4]:
delivery_time = datetime(2026, 2, 1, 12, 15)
duration = delivery_time - order_time
print("Duration:", duration)
print("Days:", duration.days)
print("Seconds component:", duration.seconds)
print("Total minutes:", duration.total_seconds() / 60)
print("Expected delivery:", order_time + timedelta(days=2))
print("Previous day:", order_time - timedelta(days=1))

Duration: 1:45:00
Days: 0
Seconds component: 6300
Total minutes: 105.0
Expected delivery: 2026-02-03 10:30:00
Previous day: 2026-01-31 10:30:00


## 3. Converting between strings and datetime
A date read as text cannot directly support date arithmetic. Parsing converts text into a date object; formatting converts a date object into text.

| Code | Meaning | Example |
|---|---|---|
| `%Y` | Four-digit year | 2026 |
| `%y` | Two-digit year | 26 |
| `%m` | Month number | 02 |
| `%d` | Day of month | 01 |
| `%H` | Hour (24-hour clock) | 14 |
| `%I` | Hour (12-hour clock) | 02 |
| `%M` | Minute | 30 |
| `%S` | Second | 00 |
| `%A`, `%B` | Full weekday, month names | Sunday, February |
| `%p` | AM/PM | PM |

### 3.1 `strftime()` — datetime to string
Think of **f** as format.

In [5]:
print(order_time.strftime("%d-%m-%Y"))
print(order_time.strftime("%A, %d %B %Y"))
print(order_time.strftime("%I:%M %p"))

01-02-2026
Sunday, 01 February 2026
10:30 AM


### 3.2 `strptime()` — string to datetime
Think of **p** as parse. The format must match the input text exactly. Specify day/month order to avoid ambiguous dates.

In [6]:
text = "05/02/2026"
parsed = datetime.strptime(text, "%d/%m/%Y")
print(parsed)
print(type(parsed))
print(parsed + timedelta(days=3))

2026-02-05 00:00:00
<class 'datetime.datetime'>
2026-02-08 00:00:00


### 3.3 `pd.to_datetime()` — converting multiple dates
Pandas can parse a list or an entire DataFrame column. ISO dates (year-month-day) are unambiguous. For other formats, specify `format`.

In [7]:
dates = ["2026-02-01", "2026-02-03", "2026-02-05"]
converted = pd.to_datetime(dates)
print(converted)
print(type(converted))
print(pd.to_datetime(["01/02/2026", "03/02/2026"], format="%d/%m/%Y"))

DatetimeIndex(['2026-02-01', '2026-02-03', '2026-02-05'], dtype='datetime64[ns]', freq=None)
<class 'pandas.core.indexes.datetimes.DatetimeIndex'>
DatetimeIndex(['2026-02-01', '2026-02-03'], dtype='datetime64[ns]', freq=None)


### 3.4 Missing and invalid dates: `NaT`
`NaT` means Not a Time. With `errors="coerce"`, invalid date strings become missing timestamps instead of raising an exception. Check them using `pd.isna()` before analysis.

In [8]:
raw_dates = ["2026-02-01", None, "2026-02-30", "2026-02-04"]
clean_dates = pd.to_datetime(raw_dates, format="%Y-%m-%d", errors="coerce")
print(clean_dates)
print(pd.isna(clean_dates))

DatetimeIndex(['2026-02-01', 'NaT', 'NaT', '2026-02-04'], dtype='datetime64[ns]', freq=None)
[False  True  True False]


## 4. Time series basics in Pandas
A basic time series is a Pandas Series whose index contains timestamps. Instead of using row numbers to identify observations, we use dates.

### 4.1 Create a Series
A shop recorded daily units sold on selected dates. Some dates are absent because no observation was recorded; absence does not automatically mean zero sales.

In [9]:
sales_dates = pd.to_datetime(["2026-01-30", "2026-01-31", "2026-02-01", "2026-02-03", "2026-02-05", "2026-02-06"])
sales = pd.Series([20, 25, 30, 40, 35, 50], index=sales_dates, name="units_sold")
sales

,units_sold
2026-01-30,20
2026-01-31,25
2026-02-01,30
2026-02-03,40
2026-02-05,35
2026-02-06,50


### 4.2 Understand the index
`DatetimeIndex` represents the collection of dates; a single element is a Pandas `Timestamp`. Pandas datetime precision is reported by the dtype and can depend on the version and input (do not assume it is always nanoseconds).

In [10]:
print(sales.index)
print(type(sales.index))
print(sales.index.dtype)
print(sales.index[0])
print(type(sales.index[0]))
print("Frequency:", sales.index.freq)

DatetimeIndex(['2026-01-30', '2026-01-31', '2026-02-01', '2026-02-03',
               '2026-02-05', '2026-02-06'],
              dtype='datetime64[ns]', freq=None)
<class 'pandas.core.indexes.datetimes.DatetimeIndex'>
datetime64[ns]
2026-01-30 00:00:00
<class 'pandas._libs.tslibs.timestamps.Timestamp'>
Frequency: None


### 4.3 Automatic alignment by dates
When two Series are added, Pandas matches index labels rather than row positions. Dates without a matching value become NaN. This example compares two shops that report on different days.

In [11]:
shop_a = sales
shop_b = pd.Series([10, 15, 20], index=pd.to_datetime(["2026-01-30", "2026-02-01", "2026-02-05"]))
display(pd.DataFrame({"shop_a": shop_a, "shop_b": shop_b, "combined": shop_a + shop_b}))

,shop_a,shop_b,combined
2026-01-30,20,10.0,30.0
2026-01-31,25,NaN,NaN
2026-02-01,30,15.0,45.0
2026-02-03,40,NaN,NaN
2026-02-05,35,20.0,55.0
2026-02-06,50,NaN,NaN


## 5. Indexing, selection and subsetting
We use `.loc` for date labels and `.iloc` for numeric positions. Sort the index before slicing dates. Date label slices include both boundaries if present.

In [12]:
print("First observation:", sales.iloc[0])
print("Sales on 3 February:", sales.loc["2026-02-03"])
print("Using a datetime object:", sales.loc[datetime(2026, 2, 3)])

First observation: 20
Sales on 3 February: 40
Using a datetime object: 40


### 5.1 Select an entire month or year
A partial date string selects all available observations in that period.

In [13]:
display(sales.loc["2026-02"])
display(sales.loc["2026"])

,units_sold
2026-02-01,30
2026-02-03,40
2026-02-05,35
2026-02-06,50


,units_sold
2026-01-30,20
2026-01-31,25
2026-02-01,30
2026-02-03,40
2026-02-05,35
2026-02-06,50


### 5.2 Select a date range
The endpoints do not need to be actual observations when the index is sorted.

In [14]:
display(sales.loc["2026-02-01":"2026-02-05"])
display(sales.loc["2026-02-02":"2026-02-04"])
display(sales.loc[datetime(2026, 2, 3):])

,units_sold
2026-02-01,30
2026-02-03,40
2026-02-05,35


,units_sold
2026-02-03,40


,units_sold
2026-02-03,40
2026-02-05,35
2026-02-06,50


### 5.3 `truncate()`
Keep observations before, after or between date boundaries. This requires a sorted index.

In [15]:
display(sales.truncate(after="2026-02-03"))
display(sales.truncate(before="2026-02-01", after="2026-02-05"))

,units_sold
2026-01-30,20
2026-01-31,25
2026-02-01,30
2026-02-03,40


,units_sold
2026-02-01,30
2026-02-03,40
2026-02-05,35


## 6. Practical: retail sales DataFrame
The following small synthetic dataset is embedded in the notebook. There are 12 transactions spanning January and February. Repeated dates represent separate sales observations, not necessarily erroneous duplicates.

### 6.1 Create and inspect the dataset

In [16]:
retail = pd.DataFrame({
    "date": ["2026-02-01", "2026-01-30", "2026-02-03", "2026-01-31",
             "2026-02-01", "2026-02-05", "2026-02-03", "2026-02-06",
             "2026-02-05", "2026-01-31", "2026-02-06", "2026-02-01"],
    "branch": ["Mathura", "Agra", "Mathura", "Agra", "Agra", "Mathura",
               "Agra", "Mathura", "Agra", "Mathura", "Agra", "Mathura"],
    "product": ["Notebook", "Pen", "Pen", "Notebook", "Pen", "Notebook",
                "Notebook", "Pen", "Pen", "Pen", "Notebook", "Pen"],
    "units_sold": [12, 20, 18, 15, 22, 25, 14, 30, 16, 10, 21, 8],
    "unit_price": [50, 10, 10, 50, 10, 50, 50, 10, 10, 10, 50, 10]
})
retail["revenue"] = retail["units_sold"] * retail["unit_price"]
display(retail)
print(retail.dtypes)

,date,branch,product,units_sold,unit_price,revenue
0,2026-02-01,Mathura,Notebook,12,50,600
1,2026-01-30,Agra,Pen,20,10,200
2,2026-02-03,Mathura,Pen,18,10,180
3,2026-01-31,Agra,Notebook,15,50,750
4,2026-02-01,Agra,Pen,22,10,220
5,2026-02-05,Mathura,Notebook,25,50,1250
6,2026-02-03,Agra,Notebook,14,50,700
7,2026-02-06,Mathura,Pen,30,10,300
8,2026-02-05,Agra,Pen,16,10,160
9,2026-01-31,Mathura,Pen,10,10,100


date          object
branch        object
product       object
units_sold     int64
unit_price     int64
revenue        int64
dtype: object


### 6.2 Convert the date column and set it as the index
`set_index("date")` makes the date the row label. `sort_index()` puts dates in chronological order. Neither operation aggregates repeated dates.

In [17]:
retail["date"] = pd.to_datetime(retail["date"], format="%Y-%m-%d")
retail_ts = retail.set_index("date").sort_index()
retail_ts

,branch,product,units_sold,unit_price,revenue
date,,,,,
2026-01-30,Agra,Pen,20,10,200
2026-01-31,Agra,Notebook,15,50,750
2026-01-31,Mathura,Pen,10,10,100
2026-02-01,Mathura,Notebook,12,50,600
2026-02-01,Agra,Pen,22,10,220
2026-02-01,Mathura,Pen,8,10,80
2026-02-03,Mathura,Pen,18,10,180
2026-02-03,Agra,Notebook,14,50,700
2026-02-05,Mathura,Notebook,25,50,1250


### 6.3 Select dates and columns in a DataFrame
For DataFrames, `.loc[row_selector, column_selector]` explicitly identifies rows and columns.

In [18]:
display(retail_ts.loc["2026-02-01"])
display(retail_ts.loc["2026-02", ["branch", "units_sold", "revenue"]])
display(retail_ts.loc["2026-02-02":"2026-02-05"])

,branch,product,units_sold,unit_price,revenue
date,,,,,
2026-02-01,Mathura,Notebook,12,50,600
2026-02-01,Agra,Pen,22,10,220
2026-02-01,Mathura,Pen,8,10,80


,branch,units_sold,revenue
date,,,
2026-02-01,Mathura,12,600
2026-02-01,Agra,22,220
2026-02-01,Mathura,8,80
2026-02-03,Mathura,18,180
2026-02-03,Agra,14,700
2026-02-05,Mathura,25,1250
2026-02-05,Agra,16,160
2026-02-06,Mathura,30,300
2026-02-06,Agra,21,1050


,branch,product,units_sold,unit_price,revenue
date,,,,,
2026-02-03,Mathura,Pen,18,10,180
2026-02-03,Agra,Notebook,14,50,700
2026-02-05,Mathura,Notebook,25,50,1250
2026-02-05,Agra,Pen,16,10,160


## 7. Time series with duplicate indices
Several purchases can occur on the same day. Duplicate index labels are valid in this situation. A lookup on a unique date in a Series returns a single value, while a repeated date returns multiple observations.

In [19]:
units = retail_ts["units_sold"]
print("Unique index?", units.index.is_unique)
print("Unique date lookup:", units.loc["2026-01-30"])
print("Repeated date lookup:")
display(units.loc["2026-02-01"])

Unique index? False
Unique date lookup: 20
Repeated date lookup:


,units_sold
date,
2026-02-01,12
2026-02-01,22
2026-02-01,8


### 7.1 Identify repeated dates
`duplicated(keep=False)` marks every observation whose index label occurs more than once.

In [20]:
retail_ts.loc[retail_ts.index.duplicated(keep=False)]

,branch,product,units_sold,unit_price,revenue
date,,,,,
2026-01-31,Agra,Notebook,15,50,750
2026-01-31,Mathura,Pen,10,10,100
2026-02-01,Mathura,Notebook,12,50,600
2026-02-01,Agra,Pen,22,10,220
2026-02-01,Mathura,Pen,8,10,80
2026-02-03,Mathura,Pen,18,10,180
2026-02-03,Agra,Notebook,14,50,700
2026-02-05,Mathura,Notebook,25,50,1250
2026-02-05,Agra,Pen,16,10,160


### 7.2 Aggregate observations sharing the same date
`groupby(level=0)` groups by the first (here, only) index level. Use `sum()` for total sales, `mean()` for average units per record, and `count()` for non-missing observations.

In [21]:
grouped = units.groupby(level=0)
daily_summary = pd.DataFrame({"total_units": grouped.sum(), "average_units": grouped.mean(), "record_count": grouped.count()})
daily_summary

,total_units,average_units,record_count
date,,,
2026-01-30,20,20.0,1
2026-01-31,25,12.5,2
2026-02-01,42,14.0,3
2026-02-03,32,16.0,2
2026-02-05,41,20.5,2
2026-02-06,51,25.5,2


### 7.3 Retail analysis: daily revenue and best sales date

In [22]:
daily_revenue = retail_ts["revenue"].groupby(level=0).sum()
display(daily_revenue)
print("Highest revenue date:", daily_revenue.idxmax().date())
print("Highest revenue:", daily_revenue.max())

,revenue
date,
2026-01-30,200
2026-01-31,850
2026-02-01,900
2026-02-03,880
2026-02-05,1410
2026-02-06,1350


Highest revenue date: 2026-02-05
Highest revenue: 1410


## 8. Quick practice (try independently)
1. Create a datetime for 16 November 2026 at 9:30 AM. Extract its month and hour.
2. Calculate the duration between 9:30 AM and 11:45 AM in minutes.
3. Parse `"18/02/2026"` and format it as `"2026-02-18"`.
4. Convert `["2026-02-01", "invalid", None]` to timestamps and identify missing entries.
5. Select the sales Series observations for January 2026.
6. Select sales between 31 January and 3 February, inclusive.
7. From `retail_ts`, retrieve all transactions on 5 February.
8. Retrieve February transactions for Mathura only.
9. Calculate average revenue per transaction for each date using `groupby(level=0)`.
10. Find the date with the largest number of transactions.

## 9. Recap
| Tool | Purpose |
|---|---|
| `datetime`, `timedelta` | Create date-time objects and durations |
| `strftime`, `strptime` | Format and parse date strings |
| `pd.to_datetime` | Convert arrays or columns into timestamps |
| `pd.isna` | Identify missing timestamps |
| `DatetimeIndex`, `Timestamp` | Represent date labels and individual dates |
| `set_index`, `sort_index` | Organize records chronologically |
| `loc`, `iloc`, `truncate` | Select temporal observations |
| `groupby(level=0)` | Summarize repeated dates |

**Reference:** Wes McKinney, *Python for Data Analysis*, 3rd edition, [11.1 Date and Time Data Types and Tools and 11.2 Time Series Basics](https://wesmckinney.com/book/time-series).